# 1. Download Final Gaze Sweeps
This cell connects to the W&B project `PCS_ET_v22` and fetches the results for sweeps:

It extracts hyperparameters, validation accuracy, test accuracy, and the `run_id`. 
The results are saved in `results/gaze_sweeps_data.csv`.

In [1]:
import wandb
import pandas as pd
import numpy as np

# 1. Setup API
api = wandb.Api()
entity = "luis-perdigao-instituto-superior-t-cnico"
project = "PCS_ET_v22"

# ---> INSERT IDs HERE <---
target_sweeps = ["4hs57q7j", "fvdarwb8", "scvkvgmm", "5fiklvgx", "zbqei1mj"] 

print(f"Connecting to {entity}/{project}...")

data_list = []

for sweep_id in target_sweeps:
    print(f"Fetching runs from sweep: {sweep_id}...")
    try:
        sweep = api.sweep(f"{entity}/{project}/{sweep_id}")
        runs = sweep.runs
    except Exception as e:
        print(f"Error accessing sweep {sweep_id}: {e}")
        continue

    print(f"  Found {len(runs)} runs. Scanning history...")

    for run in runs:
        config = {k: v for k, v in run.config.items() if not k.startswith('_')}
        
        # 2. Fetch History for both Ranking and Classification
        keys = ["accuracy_validation", "accuracy_test", "c_accuracy_test"]
        history = pd.DataFrame(run.scan_history(keys=keys))
        
        test_rank_acc = np.nan
        test_class_acc = np.nan
        
        if not history.empty and "accuracy_validation" in history.columns:
            best_epoch_idx = history["accuracy_validation"].idxmax()
            
            if "accuracy_test" in history.columns:
                test_rank_acc = history.loc[best_epoch_idx, "accuracy_test"]
            if "c_accuracy_test" in history.columns:
                test_class_acc = history.loc[best_epoch_idx, "c_accuracy_test"]
        else:
            test_rank_acc = run.summary.get("max_accuracy_test", run.summary.get("accuracy_test", np.nan))
            test_class_acc = run.summary.get("max_c_accuracy_test", run.summary.get("c_accuracy_test", np.nan))

        # 3. Extract specific parameters
        gaze_mode = config.get("gaze_mode", "off")
        attn_mode = str(config.get("attention_mode", "raw")).lower()
        seed = config.get("seed", np.nan)
        
        # Catch the "rawllout" typo just in case
        if attn_mode == 'rawllout':
            attn_mode = 'rollout'
        
        # 4. Determine method name to match your table exactly
        g_mode_lower = str(gaze_mode).lower()
        if g_mode_lower == 'align':
            method_name = f"EG-PCS-Net ({attn_mode.capitalize()})"
        elif g_mode_lower in ['off', 'disable']:
            method_name = "Baseline"
        elif g_mode_lower == 'guide':
            method_name = "GII injection"
        elif g_mode_lower == 'egvit':
            method_name = "EGViT"
        else:
            method_name = str(gaze_mode).capitalize()

        # 5. Append to data list
        data_list.append({
            "sweep_id": sweep_id,
            "run_name": run.name,
            "backbone": config.get("backbone", "unknown"),
            "gaze_method": method_name,
            "seed": seed,
            "test_rank_acc": test_rank_acc * 100 if pd.notnull(test_rank_acc) else np.nan,
            "test_class_acc": test_class_acc * 100 if pd.notnull(test_class_acc) else np.nan
        })

# 6. Save to CSV
df_new = pd.DataFrame(data_list)
filename = "gaze_modes_final.csv"
df_new.to_csv(filename, index=False)
print(f"Done! Saved {len(df_new)} runs to '{filename}'.")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: luis-perdigao (luis-perdigao-instituto-superior-t-cnico) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Connecting to luis-perdigao-instituto-superior-t-cnico/PCS_ET_v22...
Fetching runs from sweep: 4hs57q7j...
  Found 45 runs. Scanning history...
Fetching runs from sweep: fvdarwb8...
  Found 30 runs. Scanning history...
Fetching runs from sweep: scvkvgmm...
  Found 30 runs. Scanning history...
Fetching runs from sweep: 5fiklvgx...
  Found 30 runs. Scanning history...
Fetching runs from sweep: zbqei1mj...
  Found 15 runs. Scanning history...
Done! Saved 150 runs to 'gaze_modes_final.csv'.


# 2. Compute Averages and Generate Tables
This cell loads `results/gaze_sweeps_data.csv`, aggregates the runs by Method and Attention Mode, and computes the 95% Confidence Intervals.

It outputs:
1. **LaTeX Table**: Fully formatted with citations and confidence intervals.
2. **Notebook Table**: A styled Pandas DataFrame for quick viewing.

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

# --- MAPPING FUNCTIONS ---
def map_method(row):
    g_method = str(row['gaze_method'])
    if "Baseline" in g_method:
        return "Baseline"
    elif "GII injection" in g_method:
        return "GII injection \\cite{Chen2026GIIViT}"
    elif "EGViT" in g_method:
        return "EGViT \\cite{Ma2023EGViT}"
    elif "EG-PCS-Net" in g_method:
        return "EG-PCS-Net (Ours)"
    else:
        return g_method

def map_attn(row):
    g_method = str(row['gaze_method'])
    if "Raw" in g_method:
        return "Raw"
    elif "Rollout" in g_method:
        return "Rollout"
    return "--"
# -------------------------------

def generate_gaze_tables(csv_path="gaze_modes_final.csv"): 
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        return f"Error: '{csv_path}' not found. Please run Cell 1 first."

    # 1. Clean Names & Assign Method / Attention
    backbone_map = {
        'dinov3_vitb16': 'DINOv3',
        'deit3_base_patch16_224': 'DeiT III',
        'vit_base_patch16_224': 'ViT-Base',         
        'vit_base_patch16_clip_224': 'CLIP'  
    }
    df['backbone'] = df['backbone'].map(backbone_map).fillna(df['backbone'])
    df['Method'] = df.apply(map_method, axis=1)
    df['Attn Mode'] = df.apply(map_attn, axis=1)

    # 2. Aggregate (Now including 'max' for the top-1 accuracy)
    df_agg = df.groupby(['backbone', 'Method', 'Attn Mode']).agg(
        rank_mean=('test_rank_acc', 'mean'),
        rank_std=('test_rank_acc', 'std'),
        rank_max=('test_rank_acc', 'max'), # <-- NEW: Best Rank
        rank_count=('test_rank_acc', 'count'),
        class_mean=('test_class_acc', 'mean'),
        class_std=('test_class_acc', 'std'),
        class_max=('test_class_acc', 'max'), # <-- NEW: Best Class
        class_count=('test_class_acc', 'count')
    ).reset_index()

    # T-score for n=5 (4 degrees of freedom) at 95% confidence
    t_score = 2.776 
    df_agg['rank_ci'] = t_score * (df_agg['rank_std'] / np.sqrt(df_agg['rank_count']))
    df_agg['class_ci'] = t_score * (df_agg['class_std'] / np.sqrt(df_agg['class_count']))

    # 3. Create a safe String Index
    df_agg['SafeKey'] = df_agg['Method'] + "@@@" + df_agg['Attn Mode']
    
    # Pivot tables for Averages
    pivot_rank_mean = df_agg.pivot_table(index='SafeKey', columns='backbone', values='rank_mean')
    pivot_rank_ci = df_agg.pivot_table(index='SafeKey', columns='backbone', values='rank_ci')
    pivot_class_mean = df_agg.pivot_table(index='SafeKey', columns='backbone', values='class_mean')
    pivot_class_ci = df_agg.pivot_table(index='SafeKey', columns='backbone', values='class_ci')

    # Pivot tables for Max (Best Seed)
    pivot_rank_max = df_agg.pivot_table(index='SafeKey', columns='backbone', values='rank_max')
    pivot_class_max = df_agg.pivot_table(index='SafeKey', columns='backbone', values='class_max')

    # Desired Row Order mappings
    row_order = [
        ("Baseline", "--"),
        ("GII injection \\cite{Chen2026GIIViT}", "--"),
        ("EGViT \\cite{Ma2023EGViT}", "--"),
        ("EG-PCS-Net (Ours)", "Raw"),
        ("EG-PCS-Net (Ours)", "Rollout")
    ]
    
    existing_keys = []
    display_tuples = []
    for method, attn in row_order:
        k = f"{method}@@@{attn}"
        if k in pivot_rank_mean.index:
            existing_keys.append(k)
            display_tuples.append((method, attn))
            
    # Reindex all pivot tables to enforce the row order
    pivot_rank_mean = pivot_rank_mean.reindex(existing_keys)
    pivot_rank_ci = pivot_rank_ci.reindex(existing_keys)
    pivot_class_mean = pivot_class_mean.reindex(existing_keys)
    pivot_class_ci = pivot_class_ci.reindex(existing_keys)
    pivot_rank_max = pivot_rank_max.reindex(existing_keys)
    pivot_class_max = pivot_class_max.reindex(existing_keys)

    cols = ['DINOv3', 'DeiT III', 'CLIP', 'ViT-Base']
    cols = [c for c in cols if c in pivot_rank_mean.columns]

    # ==========================================================
    # TABLE 1: AVERAGE ACCURACY WITH 95% CI
    # ==========================================================
    latex_str_avg = []
    latex_str_avg.append("\\begin{table*}[ht]")
    latex_str_avg.append("\\centering")
    latex_str_avg.append("\\caption{Average Accuracy (Rank / Class) with 95\\% confidence intervals over tested seeds.}")
    latex_str_avg.append("\\label{tab:gaze_modes_avg}")
    latex_str_avg.append("\\begin{tabular}{ll" + "c"*len(cols) + "}")
    latex_str_avg.append("\\toprule")
    latex_str_avg.append(f"Method & Attention Mode & {' & '.join(cols)} \\\\")
    latex_str_avg.append("\\midrule")
    
    for safe_key in existing_keys:
        method, attn = safe_key.split("@@@")
        row_values = []
        for col in cols:
            val_r_mean = pivot_rank_mean.loc[safe_key, col]
            val_r_ci = pivot_rank_ci.loc[safe_key, col]
            val_c_mean = pivot_class_mean.loc[safe_key, col]
            val_c_ci = pivot_class_ci.loc[safe_key, col]
            
            if np.isnan(val_r_mean) or np.isnan(val_c_mean):
                row_values.append("-")
            else:
                s_r_mean_str = f"{val_r_mean:.2f}"
                s_c_mean_str = f"{val_c_mean:.2f}"
                col_max_r = pivot_rank_mean[col].max()
                col_max_c = pivot_class_mean[col].max()
                if abs(val_r_mean - col_max_r) < 1e-9: s_r_mean_str = f"\\textbf{{{s_r_mean_str}}}"
                if abs(val_c_mean - col_max_c) < 1e-9: s_c_mean_str = f"\\textbf{{{s_c_mean_str}}}"
                
                final_str = f"{s_r_mean_str}$_{{{{\\pm{val_r_ci:.2f}}}}}$/{s_c_mean_str}$_{{{{\\pm{val_c_ci:.2f}}}}}$"
                row_values.append(final_str)
        latex_str_avg.append(f"{method} & {attn} & {' & '.join(row_values)} \\\\")

    latex_str_avg.append("\\bottomrule")
    latex_str_avg.append("\\end{tabular}")
    latex_str_avg.append("\\end{table*}")

    # ==========================================================
    # TABLE 2: BEST SEED (TOP-1 ACCURACY)
    # ==========================================================
    latex_str_best = []
    latex_str_best.append("\\begin{table*}[ht]")
    latex_str_best.append("\\centering")
    latex_str_best.append("\\caption{Best Seed Accuracy (Rank / Class) across all evaluated runs.}")
    latex_str_best.append("\\label{tab:gaze_modes_best}")
    latex_str_best.append("\\begin{tabular}{ll" + "c"*len(cols) + "}")
    latex_str_best.append("\\toprule")
    latex_str_best.append(f"Method & Attention Mode & {' & '.join(cols)} \\\\")
    latex_str_best.append("\\midrule")
    
    for safe_key in existing_keys:
        method, attn = safe_key.split("@@@")
        row_values = []
        for col in cols:
            val_r_max = pivot_rank_max.loc[safe_key, col]
            val_c_max = pivot_class_max.loc[safe_key, col]
            
            if np.isnan(val_r_max) or np.isnan(val_c_max):
                row_values.append("-")
            else:
                s_r_max_str = f"{val_r_max:.2f}"
                s_c_max_str = f"{val_c_max:.2f}"
                col_max_r_best = pivot_rank_max[col].max()
                col_max_c_best = pivot_class_max[col].max()
                
                if abs(val_r_max - col_max_r_best) < 1e-9: s_r_max_str = f"\\textbf{{{s_r_max_str}}}"
                if abs(val_c_max - col_max_c_best) < 1e-9: s_c_max_str = f"\\textbf{{{s_c_max_str}}}"
                
                final_str = f"{s_r_max_str}/{s_c_max_str}"
                row_values.append(final_str)
        latex_str_best.append(f"{method} & {attn} & {' & '.join(row_values)} \\\\")

    latex_str_best.append("\\bottomrule")
    latex_str_best.append("\\end{tabular}")
    latex_str_best.append("\\end{table*}")

    # ==========================================================
    # PRINT LATEX TO CONSOLE
    # ==========================================================
    print("% " + "="*50)
    print("% LATEX TABLE 1: AVERAGES (Copy to Overleaf)")
    print("% " + "="*50)
    print("\n".join(latex_str_avg))
    print("\n")
    
    print("% " + "="*50)
    print("% LATEX TABLE 2: BEST SEED (Copy to Overleaf)")
    print("% " + "="*50)
    print("\n".join(latex_str_best))
    print("\n")

    # ==========================================================
    # PANDAS DISPLAY FOR NOTEBOOK
    # ==========================================================
    idx = pd.MultiIndex.from_tuples(display_tuples, names=["Method", "Attention Mode"])
    display_df_avg = pd.DataFrame(index=idx, columns=cols)
    display_df_best = pd.DataFrame(index=idx, columns=cols)
    
    for safe_key, (method, attn) in zip(existing_keys, display_tuples):
        for col in cols:
            # Average Data
            val_r_mean = pivot_rank_mean.loc[safe_key, col]
            val_r_ci = pivot_rank_ci.loc[safe_key, col]
            val_c_mean = pivot_class_mean.loc[safe_key, col]
            val_c_ci = pivot_class_ci.loc[safe_key, col]
            
            if np.isnan(val_r_mean) or np.isnan(val_c_mean):
                display_df_avg.loc[(method, attn), col] = "-"
            else:
                display_df_avg.loc[(method, attn), col] = f"{val_r_mean:.2f}±{val_r_ci:.2f}% / {val_c_mean:.2f}±{val_c_ci:.2f}%"
            
            # Best Seed Data
            val_r_max = pivot_rank_max.loc[safe_key, col]
            val_c_max = pivot_class_max.loc[safe_key, col]
            
            if np.isnan(val_r_max) or np.isnan(val_c_max):
                display_df_best.loc[(method, attn), col] = "-"
            else:
                display_df_best.loc[(method, attn), col] = f"{val_r_max:.2f}% / {val_c_max:.2f}%"

    print("# " + "="*50)
    print("# NOTEBOOK DISPLAY 1: AVERAGES (Rank / Class)")
    print("# " + "="*50)
    display(display_df_avg)
    
    print("\n# " + "="*50)
    print("# NOTEBOOK DISPLAY 2: BEST SEED (Rank / Class)")
    print("# " + "="*50)
    display(display_df_best)

# Run
generate_gaze_tables()

% ==================================================
% LATEX TABLE 1: AVERAGES (Copy to Overleaf)
% ==================================================
\begin{table*}[ht]
\centering
\caption{Average Accuracy (Rank / Class) with 95\% confidence intervals over tested seeds.}
\label{tab:gaze_modes_avg}
\begin{tabular}{llccc}
\toprule
Method & Attention Mode & DINOv3 & DeiT III & ViT-Base \\
\midrule
Baseline & -- & 74.61$_{{\pm1.15}}$/74.34$_{{\pm1.29}}$ & 73.78$_{{\pm0.93}}$/73.43$_{{\pm1.00}}$ & 73.36$_{{\pm1.06}}$/73.26$_{{\pm1.33}}$ \\
GII injection \cite{Chen2026GIIViT} & -- & 74.27$_{{\pm0.60}}$/74.40$_{{\pm0.78}}$ & \textbf{73.79}$_{{\pm1.07}}$/73.58$_{{\pm1.11}}$ & 72.82$_{{\pm1.19}}$/72.68$_{{\pm1.16}}$ \\
EGViT \cite{Ma2023EGViT} & -- & 74.65$_{{\pm0.77}}$/74.61$_{{\pm0.83}}$ & 73.65$_{{\pm0.86}}$/73.49$_{{\pm1.11}}$ & 73.36$_{{\pm1.04}}$/\textbf{73.36}$_{{\pm0.88}}$ \\
EG-PCS-Net (Ours) & Raw & \textbf{74.82}$_{{\pm0.78}}$/\textbf{74.79}$_{{\pm0.91}}$ & 73.65$_{{\pm0.85}}$/\text

DINOv3  \
Method                              Attention Mode                              
Baseline                            --              74.61±1.15% / 74.34±1.29%   
GII injection \cite{Chen2026GIIViT} --              74.27±0.60% / 74.40±0.78%   
EGViT \cite{Ma2023EGViT}            --              74.65±0.77% / 74.61±0.83%   
EG-PCS-Net (Ours)                   Raw             74.82±0.78% / 74.79±0.91%   
                                    Rollout         74.73±0.80% / 74.34±1.07%   

                                                                     DeiT III  \
Method                              Attention Mode                              
Baseline                            --              73.78±0.93% / 73.43±1.00%   
GII injection \cite{Chen2026GIIViT} --              73.79±1.07% / 73.58±1.11%   
EGViT \cite{Ma2023EGViT}            --              73.65±0.86% / 73.49±1.11%   
EG-PCS-Net (Ours)                   Raw             73.65±0.85% / 73.90±1.15%   
                                    Rollout         73.78±0.71% / 73.90±0.80%   

                                                                     ViT-Base  
Method                              Attention Mode                             
Baseline                            --              73.36±1.06% / 73.26±1.33%  
GII injection \cite{Chen2026GIIViT} --              72.82±1.19% / 72.68±1.16%  
EGViT \cite{Ma2023EGViT}            --              73.36±1.04% / 73.36±0.88%  
EG-PCS-Net (Ours)                   Raw             73.30±1.42% / 73.13±1.55%  
                                    Rollout         73.78±1.34% / 72.90±1.23%


# ==================================================
# NOTEBOOK DISPLAY 2: BEST SEED (Rank / Class)
# ==================================================


DINOv3  \
Method                              Attention Mode                    
Baseline                            --              76.54% / 77.05%   
GII injection \cite{Chen2026GIIViT} --              74.91% / 76.03%   
EGViT \cite{Ma2023EGViT}            --              76.54% / 76.54%   
EG-PCS-Net (Ours)                   Raw             76.20% / 76.20%   
                                    Rollout         76.71% / 76.97%   

                                                           DeiT III  \
Method                              Attention Mode                    
Baseline                            --              74.91% / 74.74%   
GII injection \cite{Chen2026GIIViT} --              76.28% / 75.68%   
EGViT \cite{Ma2023EGViT}            --              74.91% / 75.68%   
EG-PCS-Net (Ours)                   Raw             75.09% / 75.68%   
                                    Rollout         74.57% / 75.34%   

                                                           ViT-Base  
Method                              Attention Mode                   
Baseline                            --              75.60% / 75.17%  
GII injection \cite{Chen2026GIIViT} --              74.91% / 74.74%  
EGViT \cite{Ma2023EGViT}            --              74.91% / 74.57%  
EG-PCS-Net (Ours)                   Raw             76.11% / 76.20%  
                                    Rollout         76.28% / 75.43%

# 3. Generate Checkpoints Configuration
This cell automatically finds the best single run (W&B ID) for each `backbone` + `method` combination.
You can easily toggle `BEST_BY = "test"` or `"val"` to decide which metric dictates the "best" run.

It outputs the exact Python dictionary array format required for your evaluation scripts.

In [3]:
import pandas as pd
import numpy as np

# ==========================================
# CONFIGURATION
# Choose what metric determines the "best" seed
BEST_BY = "test"  # Options: "val" or "test"
# ==========================================

def generate_checkpoints_dict(csv_path="gaze_modes_final.csv", best_by="val"):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: '{csv_path}' not found. Run Cell 1 first.")
        return

    # Safety check
    if best_by == "val" and "val_rank_acc" not in df.columns:
        print("Error: 'val_rank_acc' column not found in CSV! Please update Cell 1 to save it.")
        return

    # 1. Map to simple backbone tags
    def get_simple_backbone(bb_str):
        bb_str = str(bb_str).lower()
        if 'dinov3' in bb_str: return 'dinov3'
        if 'deit' in bb_str: return 'deit'
        if 'vit' in bb_str or 'clip' in bb_str: return 'clip'
        return 'unknown'

    df['simple_bb'] = df['backbone'].apply(get_simple_backbone)

    # 2. Define combinations: (backbone tag, method tag, EXACT string in 'gaze_method' column)
    combinations = [
        ("dinov3", "baseline", "Baseline"), 
        ("dinov3", "guide", "GII injection"), 
        ("dinov3", "egvit", "EGViT"), 
        ("dinov3", "align_raw", "EG-PCS-Net (Raw)"), 
        ("dinov3", "align_rollout", "EG-PCS-Net (Rollout)"),
        
        ("clip", "baseline", "Baseline"), 
        ("clip", "guide", "GII injection"), 
        ("clip", "egvit", "EGViT"), 
        ("clip", "align_raw", "EG-PCS-Net (Raw)"), 
        ("clip", "align_rollout", "EG-PCS-Net (Rollout)"),
        
        ("deit", "baseline", "Baseline"), 
        ("deit", "guide", "GII injection"), 
        ("deit", "egvit", "EGViT"), 
        ("deit", "align_raw", "EG-PCS-Net (Raw)"), 
        ("deit", "align_rollout", "EG-PCS-Net (Rollout)")
    ]
    
    metric_col = "val_rank_acc" if best_by == "val" else "test_rank_acc"
    
    print(f"# Generating dictionary sorted by: {metric_col.upper()}")
    print("CHECKPOINTS = [")
    for bb, tag_method, method_search in combinations:
        
        subset = df[(df['simple_bb'] == bb) & (df['gaze_method'] == method_search)]
        tag = f"{bb}_{tag_method}"
        
        if subset.empty:
            best_id = "NOT_FOUND"
            val_score = 0.0
            test_score = 0.0
        else:
            # Sort by whatever metric is set in the toggle
            best_idx = subset[metric_col].idxmax()
            best_row = subset.loc[best_idx]
            
            best_id = best_row['run_name'] 
            
            # Extract both scores for the print statement
            val_score = best_row.get('val_rank_acc', 0.0)
            test_score = best_row.get('test_rank_acc', 0.0)
        
        print(f"    {{")
        print(f"        \"tag\": \"{tag}\",")
        print(f"        \"wandb_run_id\": \"{best_id}\", # val: {val_score:.2f}%, test: {test_score:.2f}%")
        print(f"        \"checkpoint\": None,")
        print(f"        \"checkpoint_kind\": \"best\",")
        print(f"    }},")
    print("]")

# Run
generate_checkpoints_dict(best_by=BEST_BY)

# Generating dictionary sorted by: TEST_RANK_ACC
CHECKPOINTS = [
    {
        "tag": "dinov3_baseline",
        "wandb_run_id": "drawn-sweep-7", # val: 0.00%, test: 76.54%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_guide",
        "wandb_run_id": "vivid-sweep-2", # val: 0.00%, test: 74.91%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_egvit",
        "wandb_run_id": "rose-sweep-12", # val: 0.00%, test: 76.54%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_align_raw",
        "wandb_run_id": "young-sweep-2", # val: 0.00%, test: 76.20%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_align_rollout",
        "wandb_run_id": "fresh-sweep-17", # val: 0.00%, test: 76.71%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "clip_baseline",
      